# 02 — Clause-aware chunking

**Purpose:** Split cleaned regulation text into clause-aware chunks (not fixed-size). Splits on section numbers, "Article", "Section", numbered clauses (e.g. 1.1, 1.2).

**Input:** `data/processed/clean_text/*.txt` + `data/raw_policies/sources_metadata.json`

**Output:** JSON chunks in `data/processed/chunked/` with metadata: `text`, `source`, `country`, `category` (and optional `source_id`, `chunk_index`, `page_start`, `page_end`). Schema: `config/chunk_schema.json`.

In [1]:
import json
import re
from pathlib import Path

PROJECT_ROOT = Path.cwd() if Path.cwd().name != 'notebooks' else Path.cwd().parent
CLEAN_DIR = PROJECT_ROOT / 'data' / 'processed' / 'clean_text'
CHUNK_DIR = PROJECT_ROOT / 'data' / 'processed' / 'chunked'
RAW_DIR = PROJECT_ROOT / 'data' / 'raw_policies'
CHUNK_DIR.mkdir(parents=True, exist_ok=True)

with open(RAW_DIR / 'sources_metadata.json') as f:
    meta = json.load(f)
sources = {Path(s['file']).stem: s for s in meta['sources']}
print(f"Source metadata for {len(sources)} documents loaded.")

Source metadata for 9 documents loaded.


## Clause-aware split

Split on: `Article X`, `Section X`, numbered headings like `1.`, `1.1`, `2.3.1`, and similar patterns. Keep each segment as one chunk; filter out very short fragments.

In [2]:
def clause_aware_split(text: str) -> list[str]:
    """Split text into clauses by common regulatory section patterns."""
    # Patterns: Article 1, Article 12; Section 1, Section 2.3; 1. Introduction, 2.1.1 Subsection
    pattern = re.compile(
        r'(?=\s*(?:Article\s+\d+|Section\s+\d+(?:\.\d+)*|'
        r'\d+\.(?:\d+\.)*\s*[A-Z]|'
        r'\n\s*\d+\.\s))',
        re.IGNORECASE
    )
    parts = pattern.split(text)
    chunks = []
    current = []
    for p in parts:
        p = p.strip()
        if not p:
            continue
        # If it looks like a heading line only, merge with next
        if len(p) < 80 and not p.endswith('.'):
            current.append(p)
            continue
        current.append(p)
        block = '\n'.join(current).strip()
        if len(block) >= 50:  # skip tiny fragments
            chunks.append(block)
        current = []
    if current:
        block = '\n'.join(current).strip()
        if len(block) >= 50:
            chunks.append(block)
    return chunks if chunks else [text.strip()] if text.strip() else []


def build_chunks_for_document(clean_path: Path, source_info: dict, source_id: str) -> list[dict]:
    """Build list of chunk dicts for one document."""
    text = clean_path.read_text(encoding='utf-8')
    raw_chunks = clause_aware_split(text)
    chunks = []
    for i, seg in enumerate(raw_chunks):
        chunks.append({
            'text': seg,
            'source': source_info.get('authority', source_id),
            'country': source_info['country'],
            'category': source_info['category'],
            'source_id': source_id,
            'chunk_index': i,
        })
    return chunks

In [3]:
# Process each clean .txt file
all_chunks = []
for txt_file in sorted(CLEAN_DIR.glob('*.txt')):
    stem = txt_file.stem
    source_info = sources.get(stem)
    if not source_info:
        print(f"No metadata for {stem}, skip.")
        continue
    doc_chunks = build_chunks_for_document(txt_file, source_info, stem)
    all_chunks.extend(doc_chunks)
    # Per-document JSON (optional)
    out_file = CHUNK_DIR / f"{stem}_chunks.json"
    with open(out_file, 'w', encoding='utf-8') as f:
        json.dump(doc_chunks, f, indent=2, ensure_ascii=False)
    print(f"{stem}: {len(doc_chunks)} chunks → {out_file.name}")

# Combined chunks file for vector index
combined_path = CHUNK_DIR / 'chunks.json'
with open(combined_path, 'w', encoding='utf-8') as f:
    json.dump(all_chunks, f, indent=2, ensure_ascii=False)
print(f"\nTotal chunks: {len(all_chunks)}. Combined: {combined_path}")

CDSCO_PV_guidance_pharma: 39 chunks → CDSCO_PV_guidance_pharma_chunks.json
EMA_GVP_Annex_I_Definitions: 26 chunks → EMA_GVP_Annex_I_Definitions_chunks.json
EMA_GVP_Module_VI: 345 chunks → EMA_GVP_Module_VI_chunks.json
FDA_postmarketing_safety_reporting: 62 chunks → FDA_postmarketing_safety_reporting_chunks.json
FDA_postmarketing_surveillance_best_practices: 74 chunks → FDA_postmarketing_surveillance_best_practices_chunks.json
FDA_risk_disclosure_promo: 16 chunks → FDA_risk_disclosure_promo_chunks.json
HIPAA_Privacy_Rule: 12 chunks → HIPAA_Privacy_Rule_chunks.json
ICH_E2D: 23 chunks → ICH_E2D_chunks.json
India_Drugs_and_Magic_Remedies_Act_1954: 52 chunks → India_Drugs_and_Magic_Remedies_Act_1954_chunks.json

Total chunks: 649. Combined: /Users/kumar/Documents/projects/new projects/Pharmaceutical Regulatory Compliance Agent/data/processed/chunked/chunks.json
